In [48]:
import yt_dlp

In [49]:
import isodate

In [50]:
from pydub import AudioSegment

In [51]:
from sarvamai import SarvamAI

In [52]:
import os

In [53]:
from dotenv import load_dotenv

load_dotenv()

True

In [54]:
client_sarvam = SarvamAI(api_subscription_key = os.getenv("SARVAM_API_KEY"))

In [55]:
client_youtube = os.getenv("YOUTUBE_API_KEY")

In [56]:
import tempfile

In [57]:
import anthropic

In [58]:
client_anthropic = anthropic.Anthropic(api_key = os.getenv('CLAUDE_API_KEY'))

In [59]:
def chunk_audio(audio_path, chunk_duration_ms=29000):
    """Split audio into 29s chunks to stay under Sarvam's limit."""
    audio = AudioSegment.from_file(audio_path)
    chunks = []

    for i, start in enumerate(range(0, len(audio), chunk_duration_ms)):
        chunk = audio[start:start + chunk_duration_ms]
        tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
        chunk.export(tmp.name, format="mp3")
        chunks.append(tmp.name)
        tmp.close()

    return chunks

In [60]:
def transcribe_audio(audio_path):
    response = client_sarvam.speech_to_text.transcribe(
        file=open(audio_path, "rb"),
        model="saaras:v3",
        mode="transcribe"
    )
    return response

In [61]:
def transcribe_long_audio(audio_path):
    chunks = chunk_audio(audio_path)
    full_transcript = []

    try:
        for i, chunk_path in enumerate(chunks):
            print(f"Transcribing chunk {i+1}/{len(chunks)}...")
            result = transcribe_audio(chunk_path)
            full_transcript.append(result.transcript)  # adjust key based on Sarvam response structure
    finally:
        # Clean up temp chunk files
        for chunk_path in chunks:
            os.remove(chunk_path)

    return " ".join(full_transcript)

In [22]:
transcript = transcribe_long_audio("/Users/Rakshit.Lodha/Desktop/audio.mp3")

transcript

Transcribing chunk 1/2...
Transcribing chunk 2/2...


"You started your company to build something real, not to drown in admin work. There's a better way to run it now. People spend months watching YouTube videos, trying random AI tools, figuring it out alone, and most of them end up right where they started. I compressed all of that into two days chat GPT and AI workshop, learned 25 plus AI tools. You build real automations, real workflows, real systems, no coding required, over 10. 10 million people have done this. Next batch this weekend, link below."

In [62]:
def download_audio(url):
    """Works for both Instagram and YouTube URLs via yt-dlp."""
    tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    tmp_path = tmp.name
    tmp.close()

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': tmp_path.replace(".mp3", ""),   # yt-dlp appends extension
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
        }],
        'quiet': True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    # yt-dlp saves as <path>.mp3, but tmp_path already ends in .mp3
    actual_path = tmp_path.replace(".mp3", "") + ".mp3"
    return actual_path

In [63]:
def process_urls(url_list):
    """
    Takes a list of Instagram/YouTube URLs.
    Returns a dict: { url -> transcript }
    """
    results = {}

    for i, url in enumerate(url_list):
        print(f"\n[{i+1}/{len(url_list)}] Processing: {url}")
        audio_path = None

        try:
            print("  Downloading audio...")
            audio_path = download_audio(url)

            print("  Transcribing...")
            transcript = transcribe_long_audio(audio_path)
            results[url] = transcript

        except Exception as e:
            print(f"Failed: {e}")
            results[url] = None

        finally:
            if audio_path and os.path.exists(audio_path):
                os.remove(audio_path)      # clean up downloaded file

    return results

In [28]:
process_urls(instagram_urls)


[1/4] Processing: https://www.instagram.com/p/DVQoG75DN8Z/?igsh=MTdzd3FhbWFxYWFjdQ==
  Transcribing...                                        
Transcribing chunk 1/2...
Transcribing chunk 2/2...

[2/4] Processing: https://www.instagram.com/p/DVQoG75DN8Z/?igsh=MTdzd3FhbWFxYWFjdQ==
  Transcribing...                                        
Transcribing chunk 1/2...
Transcribing chunk 2/2...

[3/4] Processing: https://www.instagram.com/reel/DRH4veBjY0D/?igsh=aG1yczVvaHRtbzZt
  Transcribing...                                          
Transcribing chunk 1/2...
Transcribing chunk 2/2...

[4/4] Processing: https://www.instagram.com/reel/DRJwUlOjVNR/?igsh=MXVyemM5dWFlbmpmdA==
  Transcribing...                                          
Transcribing chunk 1/3...
Transcribing chunk 2/3...
Transcribing chunk 3/3...


{'https://www.instagram.com/p/DVQoG75DN8Z/?igsh=MTdzd3FhbWFxYWFjdQ==': "You started your company to build something real, not to drown in admin work. There's a better way to run it now. People spend months watching YouTube videos, trying random AI tools, figuring it out alone, and most of them end up right where they started. I compressed all of that into two days chat GPT and AI workshop, learned 25 plus AI tools. You build real automations, real workflows, real systems, no coding required, over 10. 10 million people have done this. Next batch this weekend, link below.",
 'https://www.instagram.com/reel/DRH4veBjY0D/?igsh=aG1yczVvaHRtbzZt': "What if I told you rich Indian families secretly marry only inside four zodiac signs? India's wedding economy moves 10.7 lakh crore rupees every single year, one of the biggest private wealth transfer systems in the country. Every major astrology portal in India pushes the same categories: top signs for wealth, zodiac signs, born rich, money magnet

In [64]:
def analyze_references(urls):
    """Transcribe all URLs and extract style/structure patterns."""
    
    print("Transcribing reference videos...")
    transcripts = process_urls(urls)
    valid = {k: v for k, v in transcripts.items() if v}
    
    if not valid:
        print("No valid transcripts found.")
        return None

    reference_text = ""
    for url, transcript in valid.items():
        reference_text += f"---\nReference ({url}):\n{transcript}\n"

    print(f"\nAnalyzing {len(valid)} transcripts...")
    print(transcripts)

    prompt = f"""
You are analyzing short-form video scripts to extract creator style patterns.

Here are {len(valid)} transcripts:

{reference_text}

Extract and return a structured analysis with these sections:

1. HOOK PATTERNS
   - How do they open? (question / bold claim / stat / story?)
   - First sentence examples from the references

2. STRUCTURE
   - How is the middle organized? (list / story arc / contrast / problem-solution?)
   - Average number of points made

3. TONE & LANGUAGE
   - Formal or conversational?
   - Sentence length (short punchy vs longer flowing)
   - Any recurring phrases or patterns

4. CLOSING & CTA
   - How do they end? (soft CTA / hard sell / cliffhanger?)
   - Examples from references

5. PACING
   - Estimated words per segment
   - Total script length pattern

Be specific and pull examples from the actual transcripts.
"""

    response = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1500,
        messages=[{"role": "user", "content": prompt}]
    )

    analysis = response.content[0].text

    # Save analysis for reuse
    with open("reference_analysis.txt", "w") as f:
        f.write(analysis)
    print("\nAnalysis saved to reference_analysis.txt")

    return analysis

In [64]:
urls = ["https://youtube.com/shorts/9LOINWY_8TY?si=nPfU9hz4a4akcPta"]

In [65]:
analysis = analyze_references(urls)

Transcribing reference videos...

[1/1] Processing: https://youtube.com/shorts/9LOINWY_8TY?si=nPfU9hz4a4akcPta


  Transcribing...                                        
Transcribing chunk 1/2...
Transcribing chunk 2/2...

Analyzing 1 transcripts...
{'https://youtube.com/shorts/9LOINWY_8TY?si=nPfU9hz4a4akcPta': "Let me tell you the story of a fruit seller's son who started 40 years ago with absolutely nothing in his hand and today has built a ₹453 crore revenue generating business with 160 retail outlets. I'm talking about Raghunandan Kamat, the founder of Naturals Ice Creams. Naturals Ice Cream is rated as one of the most trusted brands in the country today, and he started from scratch and for 40 years has maintained consistency in quality and consistency in the experience for his customers. And when asked what is the secret sauce, was behind that consistency at such a big scale. He said that as a child when he used to go back home in the evenings, he had no source of entertainment. He had no television or mobile phone in his era. So the only thing he would do is he would observe the cooking me

In [65]:
def generate_script(topic, insight, analysis=None):
    """
    Generate a script using topic + insight.
    Optionally pass in saved analysis from Step 1.
    """

    # Load from file if not passed directly
    if analysis is None:
        try:
            with open("reference_analysis.txt", "r") as f:
                analysis = f.read()
            print("Loaded analysis from reference_analysis.txt")
        except FileNotFoundError:
            print("No analysis found — generating without style reference")
            analysis = "No reference analysis available. Use generic short-form video best practices."

    prompt = f"""
You are a short-form video script writer.

Here is a style analysis of reference videos from top creators:

{analysis}

Now write a NEW script using this style as a template.

TOPIC: {topic}
CORE INSIGHT: {insight}

Script structure:
1. HOOK (3-5 seconds) — stop the scroll immediately
2. CONTEXT (5 seconds) — why this matters to the viewer
3. CORE CONTENT (3-5 punchy points that deliver the insight)
4. CTA (5 seconds) — what should the viewer do next

Rules:
- Match the tone and pacing from the style analysis
- Total length: 200-250 words (fits ~90 seconds)
- No filler phrases like "In today's video..." or "Don't forget to like"
- Output the script ONLY — no labels, no commentary
"""

    response = client_anthropic.messages.create(
        model="claude-opus-4-5",
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )

    script = response.content[0].text

    # Save script
    safe_topic = topic[:30].replace(" ", "_")
    filename = f"script_{safe_topic}.txt"
    with open(filename, "w") as f:
        f.write(script)
    print(f"Script saved to {filename}")

    return script

In [45]:
topic = "Why most founders waste money on AI tools"
insight = "Most founders buy AI tools to look innovative, not to solve a real workflow problem"

script = generate_script(topic, insight)
print("\n" + "="*50)
print("GENERATED SCRIPT")
print("="*50)
print(script)

Loaded analysis from reference_analysis.txt
Script saved to script_Why_most_founders_waste_money_.txt

GENERATED SCRIPT
You're not buying AI tools. You're buying the *feeling* of being innovative.

Here's what I see every week: founders dropping $500, $800, sometimes $2,000 a month on AI subscriptions they barely open. ChatGPT Plus, Jasper, Copy.ai, Notion AI, six different image generators, three video tools. The full stack. Looks impressive in the budget spreadsheet.

But here's the problem—none of it connects to an actual workflow.

No automation. No system. No process that runs without you babysitting it.

You bought tools. You didn't solve problems.

The founders actually winning with AI? They start backwards. They identify one bottleneck—customer support taking 4 hours daily, content repurposing eating weekends, lead qualification drowning the sales team. Then they find the tool that fixes *that specific thing*.

One tool. One workflow. One result you can measure.

The rest? It's

In [66]:
from googleapiclient.discovery import build

In [67]:
from datetime import datetime, timedelta

In [68]:
API_KEY = client_youtube

In [69]:
youtube = build("youtube", "v3", developerKey=API_KEY)

In [70]:
def find_channel_id(handle):
    response = youtube.channels().list(
        forHandle = handle,
        part = "id, snippet"
    ).execute()
    
    return response['items'][0]['id']

In [71]:
find_channel_id("@Rajiv.Talreja")

'UCYEtmpGBd3jEO1A8sJ4OEMg'

In [83]:
def get_videos_data(days, channel_id):
    youtube = build("youtube", "v3", developerKey=API_KEY)
    after = (datetime.utcnow() - timedelta(days=days)).isoformat("T") + "Z"

    search = youtube.search().list(
        channelId = channel_id,
        publishedAfter = after, 
        type="video", order="date", part="id", maxResults=50
    ).execute()

    video_ids = [item["id"]["videoId"] for item in search["items"]]

    stats = youtube.videos().list(
    id=",".join(video_ids),
    part="snippet,statistics,contentDetails"
    ).execute()


    print(stats)

    for v in stats["items"]:
        return ({
            "title": v["snippet"]["title"],
            "views": v["statistics"].get("viewCount"),
            "likes": v["statistics"].get("likeCount"),
            "comments": v["statistics"].get("commentCount"),
            "link": f"https://youtube.com/watch?v={v['id']}",
            "duration_secs": isodate.parse_duration(v['contentDetails']['duration']).total_seconds()
        })

    # for v in stats["items"]:
    #     # if isodate.parse_duration(v['contentDetails']['duration']).total_seconds() < 1800:
    #         return ({
    #             "title": v["snippet"]["title"],
    #             "views": v["statistics"].get("viewCount"),
    #             "likes": v["statistics"].get("likeCount"),
    #             "comments": v["statistics"].get("commentCount"),
    #             "link": f"https://youtube.com/watch?v={v['id']}",
    #             "duration_secs": isodate.parse_duration(v['contentDetails']['duration']).total_seconds()
    #         })

In [85]:
get_videos_data(60, "UCYEtmpGBd3jEO1A8sJ4OEMg")

/var/folders/vt/2vrqs8b50f19c5dvl7krlgx80000gp/T/ipykernel_98946/3199889059.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  after = (datetime.utcnow() - timedelta(days=days)).isoformat("T") + "Z"


{'kind': 'youtube#videoListResponse', 'etag': 'Jq9sbL8JGF8hTv1hnXCKkpr_7Vo', 'items': [{'kind': 'youtube#video', 'etag': 'D4jbTrmsQq1pKvghH-dh3UcrS9s', 'id': 'pR6DJ1QZDhY', 'snippet': {'publishedAt': '2026-04-05T13:30:53Z', 'channelId': 'UCYEtmpGBd3jEO1A8sJ4OEMg', 'title': 'Humare culture me...', 'description': '', 'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/pR6DJ1QZDhY/default.jpg', 'width': 120, 'height': 90}, 'medium': {'url': 'https://i.ytimg.com/vi/pR6DJ1QZDhY/mqdefault.jpg', 'width': 320, 'height': 180}, 'high': {'url': 'https://i.ytimg.com/vi/pR6DJ1QZDhY/hqdefault.jpg', 'width': 480, 'height': 360}, 'standard': {'url': 'https://i.ytimg.com/vi/pR6DJ1QZDhY/sddefault.jpg', 'width': 640, 'height': 480}, 'maxres': {'url': 'https://i.ytimg.com/vi/pR6DJ1QZDhY/maxresdefault.jpg', 'width': 1280, 'height': 720}}, 'channelTitle': 'Rajiv Talreja', 'categoryId': '22', 'liveBroadcastContent': 'none', 'defaultLanguage': 'en-GB', 'localized': {'title': 'Humare culture me...', 'des

{'title': 'Humare culture me...',
 'views': '9248',
 'likes': '240',
 'comments': '2',
 'link': 'https://youtube.com/watch?v=pR6DJ1QZDhY',
 'duration_secs': 17.0}